#Querying Data Files

### Self-describing formats

JSON

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/students.json`; -- Einzeldatei

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/*.json`; -- wildcard -->  liest alle JSON-Dateien in diesem Ordner

Parquet

In [0]:
%sql
SELECT * 
FROM parquet.`/Volumes/workspace/default/volume/*.parquet`;

Delta

In [0]:
# Zielpfad im Volume (Delta schreibt in Ordner)
target_dir = "/Volumes/workspace/default/volume/sample_delta"

# 1) Beispiel-DataFrame
data = [
    (1, "Tesla", 50000),
    (2, "VW",  31000),
    (3, "Audi",  76000),
]
df = spark.createDataFrame(data, ["id", "name", "price"])

# 2) Als Delta ins Volume schreiben (Overwrite, falls vorhanden)
(df.write
   .format("delta")
   .mode("overwrite")
   .save(target_dir))

# Delta schreibt im Zielpfad eine "part-0000-.....parquet" Datei
# Beispiel:
#/Volumes/workspace/default/volume/part-00000-d92b98b7-b6ff-4e05-96a2-fc209758bbe5.c000.snappy.parquet

In [0]:
%sql
SELECT * 
FROM delta.`/Volumes/workspace/default/volume/sample_delta`;

## CSV (Non-self-describing formats)

In [0]:
%sql
SELECT * 
FROM csv.`/Volumes/workspace/default/volume/sample_data.csv`;

Lesen über Temporäre Views

In [0]:
%sql
-- State of the Art: Daten direkt in den Unity Catalog laden
CREATE OR REPLACE TEMPORARY VIEW sample_data
AS SELECT * FROM read_files(
  '/Volumes/workspace/default/volume/sample_data.csv',
  format => 'csv',
  header => true,
  sep => ';'
);

In [0]:
%sql
--Alternative Vairante (veraltet)
/*
CREATE OR REPLACE TEMPORARY VIEW sample_data
USING CSV
OPTIONS (
  'path'='/Volumes/workspace/default/volume/sample_data.csv',
  'delimiter'=';',
  'header'='true'
);
*/

In [0]:
%sql
SELECT * FROM sample_data;

### Ganze Verzeichnisse lesen

In [0]:
%sql
SELECT * 
FROM json.`/Volumes/workspace/default/volume/`;